# Phase 7: Final Evaluation on Holdout Test Set & Diagnostics
In this phase, we evaluate our saved, stable models on the unseen 20% holdout test set. We will calculate final performance metrics (MAE, RMSE, MAPE), visualize predictions against actuals, and perform residual diagnostics to ensure our models are not violating core regression assumptions. Finally, we will extract Feature Importances.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import numpy as np
import joblib

final_ridge_model = joblib.load("../models/ridge_stable_model.joblib")
final_lasso_model = joblib.load("../models/lasso_stable_model.joblib")
final_elasticnet_model = joblib.load("../models/elasticnet_stable_model.joblib")

print("--- Final Model Evaluation on Holdout Test Set ---")

# Değerlendirilecek modelleri bir sözlükte toplayalım
final_models = {
    "Ridge (Stable)": final_ridge_model,
    "Lasso (Stable)": final_lasso_model,
    "ElasticNet (Stable)": final_elasticnet_model
}

test_metrics = []

for name, model in final_models.items():
    # Test seti üzerinde tahmin yap
    y_pred = model.predict(X_test)

    # Metrikleri hesapla
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100 # Yüzde cinsinden

    test_metrics.append({
        "Model": name,
        "Test MAE": mae,
        "Test RMSE": rmse,
        "Test MAPE (%)": mape
    })

# Sonuçları DataFrame yapıp gösterelim
test_metrics_df = pd.DataFrame(test_metrics)

styled_test_metrics = test_metrics_df.style.background_gradient(cmap='Greens', subset=['Test MAE', 'Test RMSE', 'Test MAPE (%)'])\
                                           .format({"Test MAE": "{:.2f}", "Test RMSE": "{:.2f}", "Test MAPE (%)": "{:.2f}%"})
display(styled_test_metrics)

In [ ]:
# Görselleştirmeyi karmaşıklaştırmamak adına en iyi MAE'ye sahip modeli seçelim
best_model_name = test_metrics_df.loc[test_metrics_df['Test MAE'].idxmin(), 'Model']
best_model = final_models[best_model_name]

# Tahminleri seriye çevirelim ve test indeksini atayalım
y_test_pred = pd.Series(best_model.predict(X_test), index=y_test.index)

# Test setinden spesifik bir 1 haftalık (168 saat) pencere seçelim
# Verinin başından 1 haftalık bir kesit alıyoruz
window_size = 168
y_test_window = y_test.iloc[:window_size]
y_pred_window = y_test_pred.iloc[:window_size]

plt.figure(figsize=(16, 6))
plt.plot(y_test_window.index, y_test_window.values, label='Gerçek Talep (Actual)', color='#2C3E50', linewidth=2)
plt.plot(y_pred_window.index, y_pred_window.values, label=f'Tahmin (Predicted - {best_model_name})', color='#E74C3C', linewidth=2, linestyle='--')

plt.title(f"Test Seti Üzerinde Gerçek vs. Tahmin (1 Haftalık Kesit) - {best_model_name}", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Zaman", fontsize=12)
plt.ylabel("Toplam Talep", fontsize=12)
plt.legend(loc='best', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Kalıntıları (Gerçek - Tahmin) hesapla
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Kalıntıların Dağılımı (Histogram)
sns.histplot(residuals, kde=True, ax=axes[0], color='#3498DB', bins=50)
axes[0].set_title("Kalıntıların Dağılımı (Residual Distribution)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Hata Miktarı (Gerçek - Tahmin)", fontsize=12)
axes[0].set_ylabel("Frekans", fontsize=12)
# İdeal senaryoda bu grafik 0 etrafında normal dağılmalıdır (Çan eğrisi)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)

# 2. Tahmin Edilen Değerlere Karşı Kalıntılar (Scatter)
axes[1].scatter(y_test_pred, residuals, alpha=0.4, color='#9B59B6')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("Tahmin Değerlerine Karşı Kalıntılar (Residuals vs. Fitted)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Tahmin Edilen Talep", fontsize=12)
axes[1].set_ylabel("Kalıntılar (Residuals)", fontsize=12)
# İdeal senaryoda noktalar 0 çizgisi etrafında huni şekli oluşturmadan rastgele dağılmalıdır.

plt.tight_layout()
plt.show()

In [ ]:
# Pipeline'dan Lasso modelini ve Polynomial özellikleri çıkaralım
lasso_step = final_lasso_model.named_steps['reg']
poly_step = final_lasso_model.named_steps['poly']

# Polynomial özelliklerin isimlerini alalım
# Not: FEATURE_COLS listesini önceki adımlardan çekiyoruz
feature_names = poly_step.get_feature_names_out(input_features=FEATURE_COLS)
coefficients = lasso_step.coef_

# DataFrame'e dönüştürüp sıfır olmayanları filtreleyelim
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

# Lasso tarafından sıfırlanmayan özellikleri filtrele ve büyükten küçüğe sırala
active_features = importance_df[importance_df['Coefficient'] != 0].sort_values(by='Absolute_Importance', ascending=False)

print(f"Toplam Üretilen Özellik (Polinom dahil): {len(feature_names)}")
print(f"Lasso Tarafından Kullanılan (Sıfır Olmayan) Özellik Sayısı: {len(active_features)}")

# En önemli ilk 15 özelliği görselleştir
top_n = 15
plt.figure(figsize=(12, 8))
sns.barplot(x='Coefficient', y='Feature', data=active_features.head(top_n),
            palette=active_features.head(top_n)['Coefficient'].apply(lambda x: '#2ECC71' if x > 0 else '#E74C3C'))

plt.title(f"En Önemli {top_n} Özellik (Lasso Coefficients)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Katsayı Yönü ve Büyüklüğü (Yeşil: Pozitif Etki, Kırmızı: Negatif Etki)", fontsize=12)
plt.ylabel("Özellik Adı", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()